In [1]:
import numpy as np
import open3d as o3d
from plyfile import PlyData 
import copy
import trimesh
import os
import yaml

/Users/adeleyounis/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
with open("/Users/adeleyounis/Desktop/Capstone/wAI/config.yaml", "r") as f:
    config = yaml.safe_load(f)

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback if running in notebook or REPL
    BASE_DIR = os.getcwd()

def resolve_path(rel_path):
    return os.path.join(BASE_DIR, rel_path)

# Resolve all paths to search within repo
paths = {k: resolve_path(v) for k, v in config["paths"].items()}


In [3]:
# go from ply to txt
input_path = paths["pt_cloud_ply_path"]
output_path = paths["pt_cloud_txt_path"]

plydata = PlyData.read(input_path)  
data = np.array([list(x) for x in plydata.elements[0].data])
np.savetxt(output_path, data)

Import Point Cloud txt file

In [4]:
# might change based on what we are actually using 
pt_cloud = np.loadtxt(output_path, delimiter=' ')
points = pt_cloud[:,:3]
colors = pt_cloud[:,3:] 

print(f"original shape: {points.shape}")

# we are currently not saving colour info
# print(colors.shape)

# if colors.max() > 1.0:
#     colors = colors / 255.0

original shape: (30045, 3)


In [5]:
# generate a mesh
# make empty point cloud object
pcd = o3d.geometry.PointCloud()

# add points to the point cloud object
pcd.points = o3d.utility.Vector3dVector(points)

# downsample so faster
downpcd = pcd.voxel_down_sample(voxel_size=config["downsampling"]["voxel_size"])

downpcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
    radius=config["downsampling"]["normal_radius"], max_nn=config["downsampling"]["normal_max_nn"]))

downpcd.orient_normals_to_align_with_direction()
o3d.visualization.draw_geometries([downpcd])
print(f"Down sample shape: {len(downpcd.points)}")

Down sample shape: 17136


In [6]:
# get downsamples pts
points = np.asarray(downpcd.points)
normals = np.asarray(downpcd.normals)

inflate_amount = config["inflation"]["inflate_amount"]  # thicken up

points_new = points*inflate_amount

pcd_inflated = o3d.geometry.PointCloud()
pcd_inflated.points = o3d.utility.Vector3dVector(points_new)

pcd_inflated.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=config["downsampling"]["normal_radius"], max_nn=config["downsampling"]["normal_max_nn"]
    )
)

In [7]:
# clean the pcd
pcd_clean, ind = pcd_inflated.remove_statistical_outlier(nb_neighbors=config["cleaning"]["nb_neighbors"], 
                                                         std_ratio=config["cleaning"]["std_ratio"])

pcd_smooth = o3d.geometry.PointCloud.voxel_down_sample(pcd_clean, voxel_size=config["cleaning"]["smooth_voxel_size"])

In [15]:
pcd_smooth.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.1,  # adjust depending on cloud scale
        max_nn=30
    )
)

# Add fake back-face points to give Poisson depth
pcd_back = copy.deepcopy(pcd_smooth)
points = np.asarray(pcd_back.points)
normals = np.asarray(pcd_back.normals)

pcd_back.points = o3d.utility.Vector3dVector(points - 0.01 * normals)

# Combine
pcd_poisson = pcd_smooth + pcd_back

# Orient normals consistently
pcd_poisson.orient_normals_consistent_tangent_plane(k=200)
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_poisson,
    depth=8,      # reconstruction octree depth (higher=more detail)
    width=0,
    scale=1.1,
    linear_fit=False
)

In [16]:
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=False)


In [ ]:
# alpha shapes mesh - generalizes convex hull, can capture concavities
# pros: captures concavities, good for varied shapes.
# cons: sensitive to alpha parameter, can produce holes or disconnected components.

alpha = config["mesh"]["alpha"]  # smaller = tighter to data, larger = smoother
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd_smooth, alpha)
mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)
o3d.io.write_triangle_mesh("mackenzie.obj", mesh)


[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh

True

: 

In [9]:
center = mesh.get_center()

# translate to origin of mesh (not 0,0 in space)
mesh_centered = copy.deepcopy(mesh)
mesh_centered.translate(-center)

# mirror around Z
mesh_mirror = copy.deepcopy(mesh_centered)
mesh_mirror.transform([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

offset = np.array(config["mesh"]["mirror_offset"])
mesh_mirror.translate(offset)

mesh_centered.translate(center)
mesh_mirror.translate(center)

# Combine
mesh_combined = mesh_centered + mesh_mirror
mesh_combined.merge_close_vertices(config["mesh"]["merge_threshold"])
mesh_combined.compute_vertex_normals()

# cleanup
mesh_combined.remove_duplicated_vertices()
mesh_combined.remove_duplicated_triangles()
mesh_combined.remove_degenerate_triangles()
mesh_combined.remove_unreferenced_vertices()

o3d.visualization.draw_geometries([mesh_combined], mesh_show_back_face=True)

In [10]:
tm = trimesh.Trimesh(
    vertices=np.asarray(mesh_combined.vertices),
    faces=np.asarray(mesh_combined.triangles)
)

# fill holes
tm.fill_holes()

# back to Open3D
smoothed_mesh = o3d.geometry.TriangleMesh()
smoothed_mesh.vertices = o3d.utility.Vector3dVector(tm.vertices)
smoothed_mesh.triangles = o3d.utility.Vector3iVector(tm.faces)
smoothed_mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([smoothed_mesh], mesh_show_back_face=True)

Volume

In [11]:
# # get volume right from mesh
# voxel_size = config["voxelization"]["voxel_size"]
# voxel_grid_mesh = o3d.geometry.VoxelGrid.create_from_triangle_mesh(smoothed_mesh, voxel_size=voxel_size)

# voxels = np.asarray([v.grid_index for v in voxel_grid_mesh.get_voxels()])

# max_idx = voxels.max(axis=0) + 1
# volume = np.zeros(max_idx, dtype=np.uint8)
# volume[voxels[:,0], voxels[:,1], voxels[:,2]] = 1

# num_voxels = len(voxel_grid_mesh.get_voxels())
# volume_est = num_voxels * (voxel_size ** 3)

# print(f"Occupied voxels: {num_voxels}")
# print(f"Estimated volume: {volume_est} m^3") # average is form 0.066 to 0.9 m^3

# # create visualizer window
# vis = o3d.visualization.Visualizer()
# vis.create_window(window_name='wAI Visualize', width=800, height=600)

# vis.add_geometry(voxel_grid_mesh)
# vis.run()
# vis.destroy_window()